# Visualizations of the EEG data

This notebook is used for different basic visualisations of the EEG data

Author: Magnus Evensen, Malte Færgemann Lau $\\$
Project: Bachelor's Project - EEG Social Interaction

In [ ]:
import h5py
import numpy as np
import mne
import os

import matplotlib.pyplot as plt
import matplotlib as mpl

# mpl setup
mpl.rcParams['image.cmap'] = 'ocean'
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=plt.cm.ocean(np.linspace(0, 1, 64)))
mpl.rcParams['font.family'] = 'Arial'
mpl.rcParams['figure.figsize'] = (6, 4)
mpl.rcParams['lines.linewidth'] = 1
mpl.rcParams['figure.dpi'] = 400
%config InlineBackend.figure_format = 'retina'
%matplotlib inline
dtu_colors = {'dtured' : (0.6,0,0), 'blue': (0.1843,0.2431,0.9176), 'brightgreen' : (0.1216,0.8157,0.5098), 'navyblue' : (0.0118,0.0588,0.3098), 'yellow' : (0.9647,0.8157,0.3019), 'orange' : (0.9882,0.4627,0.2039), 'pink' : (0.9686,0.7333,0.6941), 'grey' : (0.8549,0.8549,0.8549), 'red' : (0.9098,0.2471,0.2824), 'green' : (0,0.5333,0.2078), 'purple' : (0.4745,0.1373,0.5569)}

In [ ]:
path = os.path.abspath('../FG_Data/PreprocessedEEGDATA/311A_FG_preprocessed-epo.fif')
epochs = mne.read_epochs(path, preload=True)

# Set a standard montage
montage = mne.channels.make_standard_montage('standard_1020')
epochs.set_montage(montage)
sfreq = int(epochs.info['sfreq'])
n_epochs, n_channels, n_timepoints = epochs.get_data().shape
# Band pass filter to get wave types
epochs_copy = epochs.copy() # Create copy to leave original data untouched as it is preloaded
alpha_waves = epochs_copy.filter(l_freq=8, h_freq=12, fir_design='firwin', n_jobs=12)
epochs_copy = epochs.copy()
theta_waves = epochs_copy.filter(l_freq=4, h_freq=8, fir_design='firwin', n_jobs=12)
epochs_copy = epochs.copy()
beta_waves =  epochs_copy.filter(l_freq=12, h_freq=35, fir_design='firwin', n_jobs=12)


In [ ]:
sfreq

In [ ]:
epochs.info

In [ ]:
# Plot the first four epochs
fig, axs = plt.subplots(4, 1, figsize=(10, 8))

for i, epoch in enumerate(epochs[:4]):
    # Display the epoch as a heatmap where:
    # - X-axis represents time points.
    # - Y-axis represents EEG channels.
    im = axs[i].imshow(epoch, aspect='auto', origin='lower')
    axs[i].set_title(f'Epoch {i+1}')
    axs[i].set_xlabel('Time Points')
    axs[i].set_ylabel('Channels')
    fig.colorbar(im, ax=axs[i], orientation='vertical')

plt.tight_layout()
plt.show()

In [ ]:
times = np.arange(-0.5, 5.5, 6/n_timepoints)  # time vector in seconds
# Create subplots for the four epochs (2 rows x 2 columns)
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for i, waves in enumerate([epochs.get_data()[0], theta_waves.get_data()[0], alpha_waves.get_data()[0], beta_waves.get_data()[0]]):
    ax = axes[i]
    # Plot each channel's time series as a line
    for ch in range(n_channels):
        ax.plot(times, waves[ch, :], label=f'Ch {ch+1}', alpha=0.7, lw = 1)
    ax.set_title(f"{['unfiltered', 'Theta waves', 'Alpha waves', 'Beta waves'][i]}")
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Amplitude')
    ax.set_xlim(-0.5,5.5)
    ax.set_ylim(-2.2e-5,2.2e-5)
    # Uncomment the line below if you have a small number of channels and want a legend
    # ax.legend(loc='upper right', fontsize='small')

plt.tight_layout()
plt.show()

In [ ]:

PSD, freqs = mne.time_frequency.psd_array_welch(epochs.get_data(), sfreq = sfreq, fmin=1, fmax = 40, n_fft=sfreq, n_overlap=int(sfreq*0.5), n_per_seg=sfreq, n_jobs=12, average='mean', window='hamming', output='power', verbose=None)
print(PSD.shape)
mean_psds = PSD.mean(axis=0)
fig = plt.figure(figsize= (12,5))
for j,psd in enumerate(mean_psds):
    plt.semilogy(freqs, psd, lw=1, alpha = 0.5)
plt.xlabel('Frequency (Hz)')
plt.ylabel('PSD (dB/Hz)')
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# 
# Create a figure with one subplot per epoch (2 rows x 2 columns)
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for i, waves in enumerate([epochs, theta_waves, alpha_waves, beta_waves]):
    psds = waves.compute_psd(method='welch', fmin=1, fmax=40, n_fft=500, n_overlap=int(sfreq*0.5))
    freqs = psds.freqs
    mean_psds = psds.get_data().mean(axis=0)

    ax = axes[i]
    for psd in mean_psds:
        ax.semilogy(freqs, psd, lw=1, alpha = 0.8)
    ax.set_title(f"{['unfiltered', 'Theta waves', 'Alpha waves', 'Beta waves'][i]}")
    ax.set_xlabel('Frequency (Hz)')
    ax.set_ylabel('PSD (dB/Hz)')
    ax.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
for i, waves in enumerate([epochs['T1P'], theta_waves['T1P'], alpha_waves['T1P'], beta_waves['T1P']]):
    psds = waves.compute_psd(method='welch', fmin=1, fmax=40, n_fft=sfreq, n_overlap=int(sfreq*0.5), n_jobs=12)
    psds.plot(picks='data', exclude='bads')
    plt.title(f"{['unfiltered', 'Theta waves', 'Alpha waves', 'Beta waves'][i]}")

In [ ]:
for i, waves in enumerate([alpha_waves['T1P'], alpha_waves['T1Pn']]):
    psds = waves.compute_psd(method='welch', fmin=1, fmax=40, n_fft=sfreq, n_overlap=int(sfreq*0.5), n_jobs=12)
    psds.plot(picks='data', exclude='bads')
    plt.title(f"{['feedback', 'no feedback', 'Alpha waves', 'Beta waves'][i]}")

In [ ]:
epochs.compute_psd(method='welch', fmin=1, fmax=40, n_fft=sfreq, n_overlap=int(sfreq*0.5), n_jobs=12).plot_topomap(contours = 0);

In [ ]:
def get_psds(freq_types, settings, window):
    
    overlap = window / 2
    psd_dict = {}
    
    for type in freq_types:
        waves = beta_waves if type == 'Beta' else theta_waves if type == 'Theta' else alpha_waves if type == 'Alpha' else epochs
        for setting in settings:
            powers = []
            for j in range(0,int(6/overlap-1)):
                t0 = j*overlap - 0.5
                t = window + t0 - 1/sfreq
                segment = waves[setting].copy().crop(tmin = t0, tmax = t)
                psds = segment.compute_psd(method='welch', fmin=1, fmax=40, n_fft=int(sfreq*window), n_overlap=int(sfreq*0.5*window), n_jobs=12, verbose = False); # what to do about frequency here?
                mean_psds = psds.average()
                powers.append(mean_psds.get_data().mean(axis=1))
            
            powers = np.array(powers)
            baseline = powers[5:11,:].mean()
            powers = powers/baseline # divide with baseline 
            label = f'{type}, {setting}'
            psd_dict[label] = powers

    print(f"{len(psd_dict)} items, with shape: {np.shape(psd_dict[label])}")
    return psd_dict   

In [ ]:
def roof(x):
    return int(x) + 1 if x > int(x) else int(x)

In [ ]:
def plot_TF(dict, mean = False): 
    """ Takes a dict where each value should be of size (n_segments, n_channels), \\
    and returns frequency over time plots in a (n_items, 2) grid"""
    times = np.arange(-0.5, 5.5, 6/len(next(iter(dict.values())))) # Create time points equal to powers
    fig, axes = plt.subplots(roof(len(dict)/2), 2, figsize=(12, len(dict)*1.2), constrained_layout = True)
    axes = axes.flatten()
    for i, (type, powers) in enumerate(dict.items()):
        ax = axes[i]
        
        if mean: # Plot mean with std
            mean_line = np.mean(powers, axis = 1)
            std_line = np.std(powers, axis = 1)
            ax.plot(times, mean_line, label = 'Mean power of channels', color = dtu_colors['red'])
            ax.fill_between(times, mean_line - std_line, mean_line + std_line, color=dtu_colors['red'], alpha=0.3, label='± 1 Std Dev')
        
        else: # Plot each channel
            for ch in range(n_channels):
                ax.plot(times, powers[:,ch], label=f'Ch {ch+1}', lw = 1)
            
        ax.axhline(y=1, color='black', linestyle='--', label='baseline', alpha = 0.3) # plot baseline
        ax.set_title(f"{type}")
        ax.set_ylim(0,2)
        ax.set_xlabel('Time (s)')
        ax.set_ylabel('Power')
    plt.show()

In [ ]:
window = 0.1 # What should window size be?
freq_types = ['Unfiltered', 'Theta', 'Alpha', 'Beta']
settings = ['T1P', 'T1Pn']
T1P_psds = get_psds(freq_types= freq_types, settings = settings, window= window)

In [ ]:
plot_TF(T1P_psds, mean = True)

In [ ]:
window = 0.1
freq_types = ['Unfiltered', 'Theta', 'Alpha', 'Beta']
settings = ['T3P', 'T3Pn', 'T1P', 'T1Pn', 'T23P', 'T23Pn']
some_psds = get_psds(freq_types= freq_types, settings = settings, window= window)

In [ ]:
plot_TF(some_psds, mean=True)

In [ ]:
# Compute the evoked (average across epochs)
evoked = epochs.average()

# Set a standard montage to provide sensor locations.
montage = mne.channels.make_standard_montage('standard_1020')
evoked.set_montage(montage)

# Plot a topomap at a specific time point (e.g., 0.1 seconds)

evoked.plot_topomap(times=0.1, ch_type='eeg', cmap='RdBu_r', contours=0, outlines='head', extrapolate='head', sphere=(0.0, 0.01, 0.015, 0.11));

In [ ]:
time_point = 0.1  # seconds

# Convert time point to index
time_idx = evoked.time_as_index(time_point)[0]

# Create a new figure with a custom size
fig, ax = plt.subplots(figsize=(3, 2))

# Plot the topomap manually using mne.viz.plot_topomap:
mne.viz.plot_topomap(
    evoked.data[:, time_idx],
    evoked.info,
    axes=ax,
    cmap='RdBu_r',
    contours=0,
    sphere=(0.0, 0.0, 0.01, 0.11),
    extrapolate='head'
)

plt.show()

In [ ]:
evoked.plot_sensors(kind='topomap');

In [ ]:
evoked.plot_sensors(kind='3d');